In [1]:
from diffusers import StableDiffusionPipeline, DDIMScheduler
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from PIL import Image
import numpy as np
from torch.utils.data import  Dataset, DataLoader
from torchvision.transforms.functional import to_pil_image, to_tensor
import matplotlib.pyplot as plt
from torchvision import transforms, datasets
from tqdm.notebook import tqdm
import os
import gc

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


In [33]:
class CropOrStitch:
    def __init__(self, target_size, train = False):

        self.target_size = target_size
        self.train = train

    def __call__(self, image):

        if isinstance(image, torch.Tensor):
            image = to_pil_image(image)

        width, height = image.size

        if width < self.target_size or height < self.target_size:
            repeat_h = (self.target_size // height) + 1
            repeat_w = (self.target_size // width) + 1

            stitched = torch.cat([torch.cat([to_tensor(image)] * repeat_w, dim=2)] * repeat_h, dim=1)

            image = to_pil_image(stitched[:, :self.target_size, :self.target_size])

        if self.train:
            crop_transform = transforms.RandomCrop(self.target_size)
        else:
            crop_transform = transforms.CenterCrop(self.target_size)

        return crop_transform(image)

# mean = [0.485, 0.456, 0.406]
# std = [0.229, 0.224, 0.225]

# denormalize = transforms.Lambda(lambda x: x * torch.tensor(std)[:, None, None] + torch.tensor(mean)[:, None, None])

train_transform = transforms.Compose([
    # CropOrStitch(224, train = True),
    transforms.Resize((256,256)),
    #transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    # transforms.Normalize(mean = mean, std = std)
])

test_transform = transforms.Compose([
    # CropOrStitch(224),
    transforms.Resize((256,256)),
    #transforms.CenterCrop(224),
    transforms.ToTensor(),
    # transforms.Normalize(mean = mean, std = std)
])

In [34]:
# train(sdv4)
sdv4_trainset = datasets.ImageFolder('/home/mplab/datasets/genimage/sdv1.4/train',  transform = train_transform)

# test
adm = datasets.ImageFolder('/home/mplab/datasets/genimage/adm/val', transform = test_transform)
biggan = datasets.ImageFolder('/home/mplab/datasets/genimage/biggan/val', transform = test_transform)
glide = datasets.ImageFolder('/home/mplab/datasets/genimage/glide/val', transform = test_transform)
midjourney = datasets.ImageFolder('/home/mplab/datasets/genimage/midjourney/val', transform = test_transform)
sdv4 = datasets.ImageFolder('/home/mplab/datasets/genimage/sdv1.4/val', transform = test_transform)
sdv5 = datasets.ImageFolder('/home/mplab/datasets/genimage/sdv1.5/val', transform = test_transform)
vqdm = datasets.ImageFolder('/home/mplab/datasets/genimage/vqdm/val', transform = test_transform)
wukong = datasets.ImageFolder('/home/mplab/datasets/genimage/wukong/val', transform = test_transform)

trainloader = DataLoader(sdv4_trainset, batch_size = 32, shuffle = True)
admloader = DataLoader(adm, batch_size = 32, shuffle = False)
bigganloader = DataLoader(biggan, batch_size = 32, shuffle = False)
glideloader = DataLoader(glide, batch_size = 32, shuffle = False)
midjourneyloader = DataLoader(midjourney, batch_size = 32, shuffle = False)
sdv4loader = DataLoader(sdv4, batch_size = 32, shuffle = False)
sdv5loader = DataLoader(sdv5, batch_size = 32, shuffle = False)
vqdmloader = DataLoader(vqdm, batch_size = 32, shuffle = False)
wukongloader = DataLoader(wukong, batch_size = 32, shuffle = False)

print(len(sdv4_trainset))
print(len(adm))
print(len(biggan))
print(len(glide))
print(len(midjourney))
print(len(sdv4))
print(len(sdv5))
print(len(vqdm))
print(len(wukong))

323997
12000
12000
12000
12000
12000
16000
12000
12000


In [5]:
pipe = StableDiffusionPipeline.from_pretrained("CompVis/stable-diffusion-v1-4").to(device)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

In [6]:
def mu_tilde(model, xt,x0, timestep):
    "mu_tilde(x_t, x_0) DDPM paper eq. 7"
    prev_timestep = timestep - model.scheduler.config.num_train_timesteps // model.scheduler.num_inference_steps
    alpha_prod_t_prev = model.scheduler.alphas_cumprod[prev_timestep] if prev_timestep >= 0 else model.scheduler.final_alpha_cumprod
    alpha_t = model.scheduler.alphas[timestep]
    beta_t = 1 - alpha_t
    alpha_bar = model.scheduler.alphas_cumprod[timestep]
    return ((alpha_prod_t_prev ** 0.5 * beta_t) / (1-alpha_bar)) * x0 +  ((alpha_t**0.5 *(1-alpha_prod_t_prev)) / (1- alpha_bar))*xt

def sample_xts_from_x0(model, x0, num_inference_steps=50):
    """
    Samples from P(x_1:T|x_0)
    """
    # torch.manual_seed(43256465436)
    alpha_bar = model.scheduler.alphas_cumprod
    sqrt_one_minus_alpha_bar = (1-alpha_bar) ** 0.5
    alphas = model.scheduler.alphas
    betas = 1 - alphas

    # 25.03.17 수정
    _, C, H, W = x0.size()

    timesteps = model.scheduler.timesteps.to(model.device)
    t_to_idx = {int(v):k for k,v in enumerate(timesteps)}
    xts = torch.zeros((num_inference_steps+1, C, H, W)).to(x0.device) # 25.03.17 수정
    xts[0] = x0
    for t in reversed(timesteps):
        idx = num_inference_steps-t_to_idx[int(t)]
        xts[idx] = x0 * (alpha_bar[t] ** 0.5) +  torch.randn_like(x0) * sqrt_one_minus_alpha_bar[t]


    return xts

def encode_text(model, prompts):

    text_input = model.tokenizer(
        prompts,
        padding="max_length",
        max_length=model.tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt",
    )
    with torch.no_grad():
        text_encoding = model.text_encoder(text_input.input_ids.to(model.device))[0]
    return text_encoding

def forward_step(model, model_output, timestep, sample):
    next_timestep = min(model.scheduler.config.num_train_timesteps - 2,
                        timestep + model.scheduler.config.num_train_timesteps // model.scheduler.num_inference_steps)

    # 2. compute alphas, betas
    alpha_prod_t = model.scheduler.alphas_cumprod[timestep]
    # alpha_prod_t_next = self.scheduler.alphas_cumprod[next_timestep] if next_ltimestep >= 0 else self.scheduler.final_alpha_cumprod

    beta_prod_t = 1 - alpha_prod_t

    # 3. compute predicted original sample from predicted noise also called
    # "predicted x_0" of formula (12) from https://arxiv.org/pdf/2010.02502.pdf
    pred_original_sample = (sample - beta_prod_t ** (0.5) * model_output) / alpha_prod_t ** (0.5)

    # 5. TODO: simple noising implementatiom
    next_sample = model.scheduler.add_noise(pred_original_sample,
                                    model_output,
                                    torch.LongTensor([next_timestep]))
    return next_sample


def get_variance(model, timestep): #, prev_timestep):
    prev_timestep = timestep - model.scheduler.config.num_train_timesteps // model.scheduler.num_inference_steps
    alpha_prod_t = model.scheduler.alphas_cumprod[timestep]
    alpha_prod_t_prev = model.scheduler.alphas_cumprod[prev_timestep] if prev_timestep >= 0 else model.scheduler.final_alpha_cumprod
    beta_prod_t = 1 - alpha_prod_t
    beta_prod_t_prev = 1 - alpha_prod_t_prev
    variance = (beta_prod_t_prev / beta_prod_t) * (1 - alpha_prod_t / alpha_prod_t_prev)
    return variance

def inversion_forward_process(model, x0,
                            etas = None,
                            prog_bar = False,
                            prompt = "",
                            cfg_scale = 3.5,
                            num_inference_steps=50, eps = None):

    if not prompt=="":
        text_embeddings = encode_text(model, prompt)
    uncond_embedding = encode_text(model, "")
    timesteps = model.scheduler.timesteps.to(model.device)

    # 25.03.17 수정
    _, C, H, W = x0.size()

    if etas is None or (type(etas) in [int, float] and etas == 0):
        eta_is_zero = True
        zs = None
    else:
        eta_is_zero = False
        if type(etas) in [int, float]: etas = [etas]*model.scheduler.num_inference_steps
        xts = sample_xts_from_x0(model, x0, num_inference_steps=num_inference_steps)
        alpha_bar = model.scheduler.alphas_cumprod
        zs = torch.zeros((num_inference_steps, C, H, W)).to(x0.device) # 25.03.17 수정
    t_to_idx = {int(v):k for k,v in enumerate(timesteps)}
    xt = x0
    # op = tqdm(reversed(timesteps)) if prog_bar else reversed(timesteps)
    op = tqdm(timesteps) if prog_bar else timesteps

    for t in op:
        # idx = t_to_idx[int(t)]
        idx = num_inference_steps-t_to_idx[int(t)]-1
        # 1. predict noise residual
        if not eta_is_zero:
            xt = xts[idx+1][None]
            # xt = xts_cycle[idx+1][None]

        with torch.no_grad():
            out = model.unet.forward(xt, timestep =  t, encoder_hidden_states = uncond_embedding)
            if not prompt=="":
                cond_out = model.unet.forward(xt, timestep=t, encoder_hidden_states = text_embeddings)

        if not prompt=="":
            ## classifier free guidance
            noise_pred = out.sample + cfg_scale * (cond_out.sample - out.sample)
        else:
            noise_pred = out.sample
        if eta_is_zero:
            # 2. compute more noisy image and set x_t -> x_t+1
            xt = forward_step(model, noise_pred, t, xt)

        else:
            # xtm1 =  xts[idx+1][None]
            xtm1 =  xts[idx][None]
            # pred of x0
            pred_original_sample = (xt - (1-alpha_bar[t])  ** 0.5 * noise_pred ) / alpha_bar[t] ** 0.5

            # direction to xt
            prev_timestep = t - model.scheduler.config.num_train_timesteps // model.scheduler.num_inference_steps
            alpha_prod_t_prev = model.scheduler.alphas_cumprod[prev_timestep] if prev_timestep >= 0 else model.scheduler.final_alpha_cumprod

            variance = get_variance(model, t)
            pred_sample_direction = (1 - alpha_prod_t_prev - etas[idx] * variance ) ** (0.5) * noise_pred

            mu_xt = alpha_prod_t_prev ** (0.5) * pred_original_sample + pred_sample_direction

            z = (xtm1 - mu_xt ) / ( etas[idx] * variance ** 0.5 )
            zs[idx] = z

            # correction to avoid error accumulation
            #xtm1 = mu_xt + ( etas[idx] * variance ** 0.5 )*z
            #xts[idx] = xtm1

    if not zs is None:
        zs[0] = torch.zeros_like(zs[0])

    return xt, zs, xts


def reverse_step(model, model_output, timestep, sample, eta = 0, variance_noise=None):
    # 1. get previous step value (=t-1)
    prev_timestep = timestep - model.scheduler.config.num_train_timesteps // model.scheduler.num_inference_steps
    # 2. compute alphas, betas
    alpha_prod_t = model.scheduler.alphas_cumprod[timestep]
    alpha_prod_t_prev = model.scheduler.alphas_cumprod[prev_timestep] if prev_timestep >= 0 else model.scheduler.final_alpha_cumprod
    beta_prod_t = 1 - alpha_prod_t
    # 3. compute predicted original sample from predicted noise also called
    # "predicted x_0" of formula (12) from https://arxiv.org/pdf/2010.02502.pdf
    pred_original_sample = (sample - beta_prod_t ** (0.5) * model_output) / alpha_prod_t ** (0.5)
    # 5. compute variance: "sigma_t(η)" -> see formula (16)
    # σ_t = sqrt((1 − α_t−1)/(1 − α_t)) * sqrt(1 − α_t/α_t−1)
    # variance = self.scheduler._get_variance(timestep, prev_timestep)
    variance = get_variance(model, timestep) #, prev_timestep)
    std_dev_t = eta * variance ** (0.5)
    # Take care of asymetric reverse process (asyrp)
    model_output_direction = model_output
    # 6. compute "direction pointing to x_t" of formula (12) from https://arxiv.org/pdf/2010.02502.pdf
    # pred_sample_direction = (1 - alpha_prod_t_prev - std_dev_t**2) ** (0.5) * model_output_direction
    pred_sample_direction = (1 - alpha_prod_t_prev - eta * variance) ** (0.5) * model_output_direction
    # 7. compute x_t without "random noise" of formula (12) from https://arxiv.org/pdf/2010.02502.pdf
    prev_sample = alpha_prod_t_prev ** (0.5) * pred_original_sample + pred_sample_direction

    # 8. Add noice if eta > 0
    if eta > 0:
        if variance_noise is None:
            variance_noise = torch.randn(model_output.shape, device=model.device)
        sigma_z =  eta * variance ** (0.5) * variance_noise
        prev_sample = prev_sample + sigma_z

    return prev_sample

def inversion_reverse_process(model,
                    xT,
                    etas = 0,
                    prompts = "",
                    cfg_scales = None,
                    prog_bar = False,
                    zs = None,
                    controller=None,
                    asyrp = False):

    # 수정(25.03.17)
    batch_size = max(1, len(prompts))

    text_embeddings = encode_text(model, prompts)

    # 수정(25.03.17)
    uncond_embedding = encode_text(model, "") if batch_size == 0 else encode_text(model, [""] * batch_size)

    if etas is None: etas = 0
    if type(etas) in [int, float]: etas = [etas]*model.scheduler.num_inference_steps
    assert len(etas) == model.scheduler.num_inference_steps
    timesteps = model.scheduler.timesteps.to(model.device)

    xt = xT.expand(batch_size, -1, -1, -1)
    op = tqdm(timesteps[-zs.shape[0]:]) if prog_bar else timesteps[-zs.shape[0]:]

    t_to_idx = {int(v):k for k,v in enumerate(timesteps[-zs.shape[0]:])}

    i = 0
    for t in op:

        idx = model.scheduler.num_inference_steps-t_to_idx[int(t)]-(model.scheduler.num_inference_steps-zs.shape[0]+1)
        ## Unconditional embedding

        with torch.no_grad():
            uncond_out = model.unet.forward(xt, timestep =  t,
                                            encoder_hidden_states = uncond_embedding)

            ## Conditional embedding
        if prompts:
            with torch.no_grad():
                cond_out = model.unet.forward(xt, timestep =  t,
                                                encoder_hidden_states = text_embeddings)

        z = zs[idx] if not zs is None else None
        z = z.expand(batch_size, -1, -1, -1)
        if prompts:
            ## classifier free guidance
            cfg_scales_tensor = torch.Tensor(cfg_scales).view(-1,1,1,1).to(model.device)
            noise_pred = uncond_out.sample + cfg_scales_tensor * (cond_out.sample - uncond_out.sample)
        else:
            noise_pred = uncond_out.sample

        # 2. compute less noisy image and set x_t -> x_t-1

        # if i == len(t_to_idx) - 2:
        #     print(i, 'ok')
        #     xt = reverse_step(model, noise_pred, t, xt, eta = etas[idx])

        # else:
        #      xt = reverse_step(model, noise_pred, t, xt, eta = etas[idx], variance_noise = z)

        xt = reverse_step(model, noise_pred, t, xt, eta = etas[idx], variance_noise = z)
        # xt = reverse_step(model, noise_pred, t, xt, eta = etas[idx])

        if controller is not None:
            xt = controller.step_callback(xt)

        i += 1

    return xt, zs

In [7]:
# 25.03.17 작성
@torch.no_grad()
def DDPM_invert_sample(
    pipe,
    image,
    prompt = "",
    num_inference_steps=20,
    steps = 20,
    device=device,
):

    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

    pipe.scheduler.set_timesteps(num_inference_steps)

    w0 = pipe.vae.encode(image * 2 - 1)
    w0 = 0.18215 * w0.latent_dist.sample()

    wt, zs, wts = inversion_forward_process(pipe, w0, etas=1, prompt = prompt, num_inference_steps=num_inference_steps)

    w0, _ = inversion_reverse_process(pipe, xT=wts[steps], etas=1, prog_bar=False, zs=zs[:steps])

    return w0, zs, wts    # w0는 아직 latent 공간의 텐서

In [56]:
# 복원된 이미지의 step 간 변화량 

@torch.no_grad()
def over_denosing(pipe, image, device, prompt="", num_inference_steps=20, prog_bar=False, mode='data'):

    latent, zs, wts = DDPM_invert_sample(pipe,image,prompt = "",num_inference_steps=20,steps = 20,device=device)
    # latent: 원래 이미지에 해당하는 최종 latent
    # zs: 역으로 복원한 noise 시퀀스 (z_t)
    # wts: 각 시간 스텝의 latent 값들 
    
    # 각 step마다 복원하고 디코딩
    step_latents = [latent.cpu()]   # 21개 
    step_diffs = []   # step간 latent 변화량 저장하는 리스트, 20개 

    # 디퓨전 모델의 역과정 단계별 시뮬레이션, latent 복원 
    for step in range(len(zs)):
        restored_latent, _ = inversion_reverse_process(
            pipe,
            xT=wts[step + 1],          # 현재 시점의 latent
            etas=1,                    # noise scale (DDIM 등에서 사용됨)
            prompts=prompt,            # guidance prompt
            prog_bar=False,            # 진행률 표시 여부
            zs=zs[:step + 1]           # t=0부터 현재 step까지의 noise 예측값 리스트 
)

        step_latents.append(restored_latent.cpu().detach())

        del restored_latent
        torch.cuda.empty_cache()
        gc.collect()

    # 5. 결과 반환, mode가 data면 latent 변화량 시퀀스 반환 
    if mode == 'data':
        for i in range(len(step_latents) - 1):
            step_diffs.append(step_latents[i].squeeze(0) - step_latents[i + 1].squeeze(0))
        return step_diffs

    else:
        w0 = step_latents[0].to(device) * 1 / pipe.vae.config.scaling_factor
        images = pipe.vae.decode(w0, return_dict=False)[0]
        images = (images / 2 + 0.5).clamp(0, 1)
        return images

In [58]:
step_diffs = over_denosing(pipe, image, device, num_inference_steps=20, mode='data')
print(step_diffs[0].shape)
a = torch.stack(step_diffs, dim=0)#.squeeze(1)
print(a.shape) # step간의 latent 차이 출력 

torch.Size([4, 32, 32])
torch.Size([20, 4, 32, 32])


In [102]:
from torch.utils.data import Subset

indexes = torch.randperm(6000)[:1000]
new_testset = Subset(sdv4, indexes)

print(len(new_testset))

1000


In [104]:
from torch.utils.data import Subset

# 전체 데이터 개수 (예: sdv4_trainset 길이)
total_len = len(sdv4)

# 162000 이후부터 끝까지 인덱스 생성
remaining_indexes = torch.arange(6001, total_len)

# 그 중 무작위로 30000개 선택
selected_indexes = remaining_indexes[torch.randperm(len(remaining_indexes))[:1000]]

# 새로운 Subset 생성
new_testset = Subset(sdv4, selected_indexes)

print(len(new_testset))  # 출력: 30000

1000


In [ ]:
# 라벨 매핑 (예시)
label_map = {0: 'real', 1: 'fake'}  # 필요시 조정

# 저장할 루트 디렉토리
save_train = '/home/mplab/datasets/diff/train'
save_test = '/home/mplab/datasets/diff/test/sdv4'

# 저장 경로 생성
for label in label_map.values():
    os.makedirs(os.path.join(save_test, label), exist_ok=True)

# 데이터 루프
for idx, (image, label) in tqdm(enumerate(new_testset), total=len(new_testset)):
    image = image.unsqueeze(0).to(device)

    # attention map 추출
    with torch.no_grad():
        step_diffs = over_denosing(pipe, image, device, num_inference_steps=20, mode='data')
    diffs = torch.stack(step_diffs, dim=0)
    #print(diffs.shape)

    # 저장 경로 생성
    save_dir = os.path.join(save_test, label_map[label])
    save_path = os.path.join(save_dir, f'image{idx}.pt')

    torch.save(diffs.cpu(), save_path)

    if (idx+1) % 1000 == 0 or idx == 0:
        print(f'Saved: {save_path}')

    del step_diffs, diffs
    torch.cuda.empty_cache()
    gc.collect()


  0%|          | 0/1000 [00:00<?, ?it/s]

Saved: /home/mplab/datasets/diff/test/sdv4/fake/image0.pt


In [ ]:
import os
import glob
import torch
from torch.utils.data import Dataset

class LatentDiffDataset(Dataset):
    def __init__(self, root_dir):
        super().__init__()
        self.files = sorted(glob.glob(os.path.join(root_dir, "*.pt")))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data = torch.load(self.files[idx])  # 저장 구조: {"x": [T, C, H, W], "label": int}
        x = data["x"]                        # [20, 4, 32, 32] — step_diffs
        label = data["label"]               # 정수 클래스
        return x, label

trainset = LatentDiffDataset("/path/to/train_pt_files")
trainloader = torch.utils.data.DataLoader(trainset, batch_size=16, shuffle=True, num_workers=2)

testset = LatentDiffDataset("/path/to/test_pt_files")
testloader = torch.utils.data.DataLoader(testset, batch_size=16, shuffle=False, num_workers=2)

In [ ]:
# Multi Head Self Attention 블록 
import torch
import torch.nn as nn

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads=8):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):  # x: [N, T, D]
        attn_out, _ = self.attn(x, x, x)
        return self.norm(x + attn_out)

class TemporalAggregation(nn.Module):
    def __init__(self, T, D, num_heads=8, num_layers=2):
        super().__init__()
        self.attn_blocks = nn.Sequential(*[
            MultiHeadSelfAttention(D, num_heads) for _ in range(num_layers)
        ])

    def forward(self, x):  # x: [B, T, H, W]
        B, T, H, W = x.shape
        x = x.permute(0, 2, 3, 1).reshape(B * H * W, T, 1)  # [B*H*W, T, D=1]
        out = self.attn_blocks(x)                          # [B*H*W, T, D]
        last = out[:, -1, :]                               # [B*H*W, D]
        return last.view(B, H, W, 1)                        # [B, H, W, 1]
        
class SpatialFocusing(nn.Module):
    def __init__(self, H, W, D=1, num_heads=8, num_layers=2):
        super().__init__()
        self.attn_blocks = nn.Sequential(*[
            MultiHeadSelfAttention(D, num_heads) for _ in range(num_layers)
        ])
        self.conv1x1 = nn.Conv1d(D, 1, kernel_size=1)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):  # x: [B, T, H, W]
        B, T, H, W = x.shape
        pooled = (x.mean(dim=1) + x.max(dim=1)[0]) / 2  # [B, H, W]
        pooled = pooled.view(B, H * W, 1).expand(-1, -1, D)  # [B, H*W, D]
        x = self.attn_blocks(pooled)  # [B, H*W, D]
        x = x.permute(0, 2, 1)        # [B, D, H*W]
        weights = self.softmax(self.conv1x1(x))  # [B, 1, H*W]
        return weights.view(B, 1, H, W).permute(0, 2, 3, 1)  # [B, H, W, 1]

temporal_module = TemporalAggregation(T=20, D=4)
spatial_module = SpatialFocusing(H=32, W=32, D=4)

# 입력
x = torch.stack(step_diffs).unsqueeze(0)  # [1, 20, 4, 32, 32]

# Temporal Attention
T_feat = x.permute(0, 2, 3, 4, 1)  # [1, C=4, H=32, W=32, T=20]
T_feat = T_feat.reshape(1 * 32 * 32, 20, 4)  # [1024, 20, 4]
T_out = temporal_module.attn_blocks(T_feat)  # [1024, 20, 4]
T_last = T_out[:, -1, :].view(1, 32, 32, 4)  # [1, 32, 32, 4]

# Spatial Attention
F_prime = spatial_module(x.squeeze(0))  # [1, 32, 32, 4]

# 최종 Feature
T_final = T_last * F_prime              # [1, 32, 32, 4]

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet18

class ResNet18_4ch(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.model = resnet18(pretrained=False)
        
        # 입력 채널 4개에 맞게 첫 conv 수정
        self.model.conv1 = nn.Conv2d(
            in_channels=4, out_channels=64,
            kernel_size=7, stride=2, padding=3, bias=False
        )
        
        # 분류기 출력 수정
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)

    def forward(self, x):  # x: [B, H, W, C=4]
        x = x.permute(0, 3, 1, 2)  # → [B, C, H, W]
        return self.model(x)

In [ ]:
model = ResNet18_4ch(num_classes=2)

T_final = torch.randn(8, 32, 32, 4)  # 예시 입력
logits = model(T_final)             # [8, 2]
print(logits.shape)

In [ ]:
class AttentionClassifier(nn.Module):
    def __init__(self, temporal_module, spatial_module, classifier):
        super().__init__()
        self.temporal = temporal_module
        self.spatial = spatial_module
        self.classifier = classifier

    def forward(self, x):  # x: [B, T=20, C=4, H=32, W=32]
        B = x.size(0)
        # Temporal Attention
        xt = x.permute(0, 2, 3, 4, 1).reshape(B * 32 * 32, 20, 4)  # [B*32*32, 20, 4]
        T_out = self.temporal.attn_blocks(xt)                     # [B*32*32, 20, 4]
        T_last = T_out[:, -1, :].view(B, 32, 32, 4)               # [B, 32, 32, 4]

        # Spatial Attention
        F_prime = self.spatial(x)                                 # [B, 32, 32, 4]

        # Pointwise 곱
        T_final = T_last * F_prime                                # [B, 32, 32, 4]

        return self.classifier(T_final)                           # [B, num_classes]


In [ ]:
model = AttentionClassifier(temporal_module, spatial_module, ResNet18_4ch())
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(num_epochs):
    model.train()
    for x, label in trainloader:
        x, label = x.to(device), label.to(device)
        out = model(x)  # [B, num_classes]
        loss = criterion(out, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    print(f"[Epoch {epoch+1}] Loss: {loss.item():.4f}")